# Generative AI as a Modeling Partner

This notebook is written for **business and analytics students**, not programmers. Most of the "AI" content here is **text you read** — the point is to practice **validation**, not prompting tricks.

This notebook demonstrates three practical roles that generative AI (LLMs like ChatGPT or Claude) can play in a prescriptive analytics workflow — and what human validation looks like at each step.

Understanding this is critical because:
- **GenAI accelerates messy work** — translating business problems into model structure, explaining results, drafting communications
- **GenAI is not an authority** — it can be fluent and wrong at the same time
- **The same judgment checkpoints from Lesson 3 apply:** Does this match the real business problem? Would I stake my reputation on it?
- **Professional skill:** using GenAI well means building a fast review discipline — verify, then use

**What you will run in code:** a small optimization model so you have real numbers to compare against an LLM-style explanation.


## Key Concepts

**Three practical roles for GenAI in prescriptive analytics:**

1. **Formulation assistant:** Turn a messy English description into candidate decision variables, objective, and constraints
2. **Interpretation assistant:** Explain solver output in business language — *must be checked against the numbers*
3. **Communication assistant:** Draft stakeholder messages — *must be checked for accuracy, missing caveats, and tone*

**The validation rule:**  
LLMs can confidently produce wrong formulations, misleading interpretations, and polished-but-inaccurate narratives. **No GenAI output should go to a decision-maker without human review.**

**About this notebook:** we do not call a live LLM. The "LLM responses" are simulated text blocks so every classroom can run the same example.


## Scenario: Call Center Staffing Optimization

A regional insurance company needs to optimize staffing across three call center shifts:
morning (6am-2pm), afternoon (2pm-10pm), and overnight (10pm-6am).

You have been asked to:
1. Build an optimization model to minimize total staffing cost while meeting service requirements
2. Interpret the model output for the operations team
3. Draft an explanation for the VP of Customer Experience

You will use a GenAI assistant for all three tasks — and validate each output before using it.

## Step 1: Install Required Packages

**What you are doing:** installing `pulp` (to solve the staffing optimization) plus plotting/table utilities.

Even in a "GenAI" lesson, prescriptive analytics still grounds itself in **a real model** — otherwise interpretation practice is imaginary.


In [1]:
%pip install numpy matplotlib pandas pulp -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Import Libraries

**Standard setup:** `numpy`, `pandas`, `matplotlib` for small numerical summaries and charts if needed, and `pulp` for the optimization.

**`np.random.seed(42)`** is included for consistency with other Lesson 11 notebooks (this notebook is mostly deterministic).


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pulp import LpMinimize, LpProblem, LpVariable, LpInteger, value, PULP_CBC_CMD
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

Libraries loaded.


## Role 1: Formulation Assistant

**Story:** you describe a staffing situation in plain English. A GenAI tool proposes a structured optimization model.

**Notebook flow for Role 1:**
- **The next code cell** prints a **simulated** LLM formulation (stored in a Python dictionary) so everyone sees the same classroom example.
- **The code cell after that** prints a **human validation checklist** — the skill you are practicing is marking items as correct, missing, or invented.

**Your job as a student:** read each proposed variable and constraint and ask:
- Does this match how *our* company actually schedules people?
- Did it invent numbers or policies that are not true?
- Did it miss a real option (like an allowed shift combination)?


In [3]:
# === YOUR PROMPT TO THE LLM ===
prompt = """
I need to staff a call center across three shifts: morning, afternoon, and overnight.
Each shift needs a minimum number of agents to meet our service level agreement (SLA).
We want to minimize total labor cost. Agents can work any shift but cost different amounts
depending on when they work. Some agents work across two consecutive shifts.
Help me set up an optimization model.
"""

# === SIMULATED LLM RESPONSE ===
llm_formulation = {
    'Decision variables': [
        'x_m  = number of agents assigned to morning shift only',
        'x_a  = number of agents assigned to afternoon shift only',
        'x_n  = number of agents assigned to overnight shift only',
        'x_ma = number of agents working morning + afternoon (double shift)',
        'x_an = number of agents working afternoon + overnight (double shift)',
    ],
    'Objective': 'Minimize: cost_m*x_m + cost_a*x_a + cost_n*x_n + cost_ma*x_ma + cost_an*x_an',
    'Constraints': [
        'Morning coverage:   x_m + x_ma >= min_morning',
        'Afternoon coverage: x_a + x_ma + x_an >= min_afternoon',
        'Overnight coverage: x_n + x_an >= min_overnight',
        'All variables >= 0 and integer',
        'Maximum double shifts per day: x_ma + x_an <= budget_doubles  [suggested: 5]',
    ]
}

print('=== LLM FORMULATION SUGGESTION ===')
print()
print('Decision Variables:')
for v in llm_formulation['Decision variables']:
    print(f'  {v}')
print()
print('Objective:')
print(f"  {llm_formulation['Objective']}")
print()
print('Constraints:')
for c in llm_formulation['Constraints']:
    print(f'  {c}')

=== LLM FORMULATION SUGGESTION ===

Decision Variables:
  x_m  = number of agents assigned to morning shift only
  x_a  = number of agents assigned to afternoon shift only
  x_n  = number of agents assigned to overnight shift only
  x_ma = number of agents working morning + afternoon (double shift)
  x_an = number of agents working afternoon + overnight (double shift)

Objective:
  Minimize: cost_m*x_m + cost_a*x_a + cost_n*x_n + cost_ma*x_ma + cost_an*x_an

Constraints:
  Morning coverage:   x_m + x_ma >= min_morning
  Afternoon coverage: x_a + x_ma + x_an >= min_afternoon
  Overnight coverage: x_n + x_an >= min_overnight
  All variables >= 0 and integer
  Maximum double shifts per day: x_ma + x_an <= budget_doubles  [suggested: 5]


In [4]:
# === HUMAN VALIDATION OF LLM FORMULATION ===

validation_notes = [
    {'item': 'Decision variables x_m, x_a, x_n',
     'verdict': 'CORRECT',
     'note': 'These match our actual staffing structure.'},
    {'item': 'Double-shift variables x_ma, x_an',
     'verdict': 'CORRECT',
     'note': 'Good — LLM correctly inferred these from our description.'},
    {'item': 'Objective function',
     'verdict': 'CORRECT',
     'note': 'Cost minimization is the right objective.'},
    {'item': 'Coverage constraints',
     'verdict': 'CORRECT',
     'note': 'Correctly accounts for agents who cover multiple shifts.'},
    {'item': 'Maximum double shifts constraint (budget_doubles = 5)',
     'verdict': 'NEEDS REVISION',
     'note': 'LLM invented this limit. Our actual policy allows up to 8 double shifts per day.'},
    {'item': 'Missing: mn_double (morning + overnight) variable',
     'verdict': 'MISSING',
     'note': 'LLM omitted the morning-overnight double shift, which is allowed but rarely used.'},
]

print('=== HUMAN VALIDATION: FORMULATION REVIEW ===')
print()
for v in validation_notes:
    print(f"[{v['verdict']:<14}]  {v['item']}")
    print(f"                  Note: {v['note']}")
    print()
print('Verdict: LLM formulation is 80% correct. Two issues must be fixed before use.')
print('This is typical — GenAI is a good starting point, not a finished model.')

=== HUMAN VALIDATION: FORMULATION REVIEW ===

[CORRECT       ]  Decision variables x_m, x_a, x_n
                  Note: These match our actual staffing structure.

[CORRECT       ]  Double-shift variables x_ma, x_an
                  Note: Good — LLM correctly inferred these from our description.

[CORRECT       ]  Objective function
                  Note: Cost minimization is the right objective.

[CORRECT       ]  Coverage constraints
                  Note: Correctly accounts for agents who cover multiple shifts.

[NEEDS REVISION]  Maximum double shifts constraint (budget_doubles = 5)
                  Note: LLM invented this limit. Our actual policy allows up to 8 double shifts per day.

[MISSING       ]  Missing: mn_double (morning + overnight) variable
                  Note: LLM omitted the morning-overnight double shift, which is allowed but rarely used.

Verdict: LLM formulation is 80% correct. Two issues must be fixed before use.
This is typical — GenAI is a good starting p

## Role 2: Interpretation Assistant

**Story:** after humans correct the formulation, we solve a staffing model with concrete costs and minimum coverage rules. Then we compare a narrative "explanation" to the **actual** solver output.

**Notebook flow for Role 2:**
- **The next code cell** builds and solves the optimization: integer variables for single-shift and double-shift patterns, minimum coverage by shift, a cap on double shifts, and a total labor cost objective.
- **The code cell after that** prints a **simulated** LLM interpretation, then prints **verification prompts** so you compare the story to the numbers you just produced.

**Red flag to learn:** smooth language can sound authoritative even when details are wrong — always tie claims back to the solver output (counts, zeros vs nonzeros, which constraints bind).


In [5]:
# === BUILD AND SOLVE THE CORRECTED MODEL ===
# Staffing minimums (agents needed per shift to meet SLA)
min_morning   = 12
min_afternoon = 18
min_overnight = 8

# Labor costs per shift
cost_m   = 180   # morning only
cost_a   = 200   # afternoon only
cost_n   = 240   # overnight only (premium)
cost_ma  = 320   # morning + afternoon double
cost_an  = 380   # afternoon + overnight double
cost_mn  = 370   # morning + overnight double (rare)

# Build model
model = LpProblem('CallCenter_Staffing', LpMinimize)
x_m   = LpVariable('morning',     lowBound=0, cat=LpInteger)
x_a   = LpVariable('afternoon',   lowBound=0, cat=LpInteger)
x_n   = LpVariable('overnight',   lowBound=0, cat=LpInteger)
x_ma  = LpVariable('morning_aft', lowBound=0, cat=LpInteger)
x_an  = LpVariable('aft_night',   lowBound=0, cat=LpInteger)
x_mn  = LpVariable('morn_night',  lowBound=0, cat=LpInteger)

# Objective
model += (cost_m*x_m + cost_a*x_a + cost_n*x_n +
          cost_ma*x_ma + cost_an*x_an + cost_mn*x_mn)

# Coverage constraints
model += x_m + x_ma + x_mn >= min_morning,   'Morning_SLA'
model += x_a + x_ma + x_an >= min_afternoon, 'Afternoon_SLA'
model += x_n + x_an + x_mn >= min_overnight, 'Overnight_SLA'

# Double-shift policy: max 8 per day
model += x_ma + x_an + x_mn <= 8, 'Double_shift_limit'

model.solve(PULP_CBC_CMD(msg=0))

# Extract results
results = {
    'Morning only':          int(value(x_m)),
    'Afternoon only':        int(value(x_a)),
    'Overnight only':        int(value(x_n)),
    'Morning+Afternoon':     int(value(x_ma)),
    'Afternoon+Overnight':   int(value(x_an)),
    'Morning+Overnight':     int(value(x_mn)),
}
total_cost = int(value(model.objective))

print('=== OPTIMIZATION RESULTS ===')
for role, count in results.items():
    print(f'  {role:<25}: {count} agents')
print(f'  {"Total daily labor cost":<25}: ${total_cost:,}')

=== OPTIMIZATION RESULTS ===
  Morning only             : 12 agents
  Afternoon only           : 10 agents
  Overnight only           : 0 agents
  Morning+Afternoon        : 0 agents
  Afternoon+Overnight      : 8 agents
  Morning+Overnight        : 0 agents
  Total daily labor cost   : $7,200


In [6]:
# === SIMULATED LLM INTERPRETATION ===
llm_interpretation = """
The model recommends a lean staffing strategy that heavily leverages double-shift agents
to cover the high-demand afternoon shift. The optimizer chose not to use morning+overnight
doubles because that combination is expensive relative to its coverage benefit.

The binding constraint is the overnight minimum — the model staffed exactly 8 overnight
agents, using a combination of overnight-only and afternoon+overnight doubles.
If you need more overnight flexibility, relaxing this minimum would reduce cost.

One concern: the model assigns zero morning+overnight doubles, suggesting the model
finds it cheaper to combine dedicated morning agents with affordable afternoon staff
than to use the more expensive morning+overnight double rate.
"""

print('=== LLM INTERPRETATION OF MODEL OUTPUT ===')
print(llm_interpretation)

# === HUMAN VALIDATION OF INTERPRETATION ===
print('=== HUMAN VALIDATION: INTERPRETATION REVIEW ===')
print()

actual_doubles = results['Morning+Afternoon'] + results['Afternoon+Overnight'] + results['Morning+Overnight']
print(f'Check 1 — Double shifts used: {actual_doubles} (limit is 8)')
print(f'  LLM said: "heavily leverages double-shift agents" -- ', end='')
print('VERIFY against actual results above.')
print()
print('Check 2 — Binding constraint claim:')
print(f'  Overnight minimum required: {min_overnight}')
overnight_covered = results['Overnight only'] + results['Afternoon+Overnight'] + results['Morning+Overnight']
print(f'  Overnight coverage in solution: {overnight_covered}')
print('  LLM should be checked: is overnight actually the binding constraint?')
print()
print('Check 3 — LLMs sometimes explain results that feel logical but do not reflect')
print('  the actual solver mathematics. Always verify key claims numerically.')
print()
print('Rule: if an explanation sounds too clean, or explains something you did not expect,')
print('confirm it against the model data before passing it to a stakeholder.')

=== LLM INTERPRETATION OF MODEL OUTPUT ===

The model recommends a lean staffing strategy that heavily leverages double-shift agents
to cover the high-demand afternoon shift. The optimizer chose not to use morning+overnight
doubles because that combination is expensive relative to its coverage benefit.

The binding constraint is the overnight minimum — the model staffed exactly 8 overnight
agents, using a combination of overnight-only and afternoon+overnight doubles.
If you need more overnight flexibility, relaxing this minimum would reduce cost.

One concern: the model assigns zero morning+overnight doubles, suggesting the model
finds it cheaper to combine dedicated morning agents with affordable afternoon staff
than to use the more expensive morning+overnight double rate.

=== HUMAN VALIDATION: INTERPRETATION REVIEW ===

Check 1 — Double shifts used: 8 (limit is 8)
  LLM said: "heavily leverages double-shift agents" -- VERIFY against actual results above.

Check 2 — Binding constrain

## Role 3: Communication Assistant

**Story:** executives need concise language, but conciseness is not allowed to smuggle in **overconfidence** or **missing assumptions**.

**Notebook flow for Role 3:**
- **The next code cell** prints a polished email-style draft (simulated LLM output), then prints a **communication review checklist** (accuracy, missing numbers, missing risks, tone, and overclaims).

**Teaching point:** the optimization model enforces minimum staffing — it does not, by itself, "ensure" customer happiness. Language matters ethically and professionally.


In [7]:
# === SIMULATED LLM COMMUNICATION DRAFT ===
llm_communication_draft = """
Subject: Staffing Recommendation — Call Center Shifts

Our optimization model recommends a staffing plan that fully meets all three shift service
level targets while reducing daily labor cost compared to our current schedule. The plan
uses a mix of single-shift and double-shift assignments, prioritizing cost-effective
coverage of our highest-volume afternoon shift. All minimum staffing levels are met or
exceeded, ensuring customers experience consistent response times throughout the day.

We recommend implementing this schedule for the next scheduling cycle and reviewing
outcomes after four weeks.
"""

print('=== LLM COMMUNICATION DRAFT ===')
print(llm_communication_draft)

# === HUMAN VALIDATION OF COMMUNICATION ===
communication_checks = [
    {'check': 'Accuracy — does it match the actual recommendation?',
     'verdict': 'MOSTLY ACCURATE',
     'note': 'Correct that SLAs are met. "Reducing cost" needs to be verified against current schedule cost.'},
    {'check': 'Missing: specific numbers',
     'verdict': 'ADD THIS',
     'note': 'VP will likely want the actual cost and agent counts, not just "cost-effective".'},
    {'check': 'Missing: key assumptions and risks',
     'verdict': 'ADD THIS',
     'note': 'Draft omits that the model assumes constant call volume. Seasonal spikes may require adjustment.'},
    {'check': 'Tone appropriate for VP audience?',
     'verdict': 'YES',
     'note': 'Non-technical, outcome-focused language is appropriate.'},
    {'check': 'Overconfident claim: "ensuring customers experience consistent response times"',
     'verdict': 'SOFTEN',
     'note': 'Model minimizes cost subject to minimum staffing — it does not guarantee service quality.'},
]

print('=== HUMAN VALIDATION: COMMUNICATION REVIEW ===')
print()
for c in communication_checks:
    print(f"[{c['verdict']:<14}]  {c['check']}")
    print(f"                  {c['note']}")
    print()
print('Next step: revise the draft to add specific numbers, acknowledge assumptions,')
print('and soften the service quality claim before sending to the VP.')

=== LLM COMMUNICATION DRAFT ===

Subject: Staffing Recommendation — Call Center Shifts

Our optimization model recommends a staffing plan that fully meets all three shift service
level targets while reducing daily labor cost compared to our current schedule. The plan
uses a mix of single-shift and double-shift assignments, prioritizing cost-effective
coverage of our highest-volume afternoon shift. All minimum staffing levels are met or
exceeded, ensuring customers experience consistent response times throughout the day.

We recommend implementing this schedule for the next scheduling cycle and reviewing
outcomes after four weeks.

=== HUMAN VALIDATION: COMMUNICATION REVIEW ===

[MOSTLY ACCURATE]  Accuracy — does it match the actual recommendation?
                  Correct that SLAs are met. "Reducing cost" needs to be verified against current schedule cost.

[ADD THIS      ]  Missing: specific numbers
                  VP will likely want the actual cost and agent counts, not just "co

## Summary: The Validation Rule in Practice

**What this section is for:** zoom out and compare the three roles side by side.

You should leave this notebook able to say, in your own words, **what you would verify first** for each role before sending anything onward.


In [8]:
# === VALIDATION SUMMARY ===
summary_data = {
    'Role': ['Formulation', 'Interpretation', 'Communication'],
    'GenAI Accuracy': ['~80%', '~70%', '~75%'],
    'Key Error': [
        'Invented a constraint limit; missed one variable',
        'Claims may not match solver output exactly',
        'Overstated certainty; omitted key assumptions',
    ],
    'Validation Required': ['Yes', 'Yes', 'Yes'],
}
df_summary = pd.DataFrame(summary_data)
print('=== GENAI PERFORMANCE ACROSS THREE ROLES ===')
print(df_summary.to_string(index=False))
print()
print('Observation: GenAI was useful in every role. It was also wrong in every role.')
print('The professional value of GenAI comes from knowing where to verify — not from trusting the output.')

=== GENAI PERFORMANCE ACROSS THREE ROLES ===
          Role GenAI Accuracy                                        Key Error Validation Required
   Formulation           ~80% Invented a constraint limit; missed one variable                 Yes
Interpretation           ~70%       Claims may not match solver output exactly                 Yes
 Communication           ~75%    Overstated certainty; omitted key assumptions                 Yes

Observation: GenAI was useful in every role. It was also wrong in every role.
The professional value of GenAI comes from knowing where to verify — not from trusting the output.


## Key Takeaways (Outro)

1. **Formulation assistant:** fast first draft — verify structure against real policies and real decisions.

2. **Interpretation assistant:** helpful storytelling — verify every important claim against solver outputs.

3. **Communication assistant:** fast drafting — verify accuracy, add specifics, add assumptions, soften overclaims.

4. **Validation is not optional** — it is the core professional competence when using GenAI in analytics.

5. **Fluency ≠ truth.** The danger is not that LLMs are useless; the danger is that they are **persuasive**.

**Next notebook:** ethical considerations — when data, objectives, and omitted constraints create harm even in "automated" systems.
